# ADANI PORTS: Advanced Time Series Analysis
This notebook uses our modular `analyzer` package to perform professional data cleaning, 
statistical stationarity tests, and signal decomposition.

In [ ]:
import sys, os
import pandas as pd

# 1. Clear path and re-add
sys.path.append(os.path.abspath("../src"))

# 2. Import modules
from analyzer.preprocessor import fill_gaps, handle_outliers

# 3. Load Data Freshly
df_raw = pd.read_csv("../data/ADANIPORTS.csv")

# 4. Check what columns we ACTUALLY have before starting
print("Columns in file:", df_raw.columns.tolist())

# 5. Process
try:
    # Use the new robust function
    df = fill_gaps(df_raw, target_col='Close')
    df = handle_outliers(df, column='Close')
    print("\nSuccess! Data is cleaned and gaps are filled.")
    print("New Columns:", df.columns.tolist())
    display(df.head())
except Exception as e:
    print(f"\nProcessing failed: {e}")

Columns in file: ['Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble']

Success! Data is cleaned and gaps are filled.
New Columns: ['Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble']


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2007-11-27,MUNDRAPORT,EQ,440.00,770.000000,1050.000000,770.000000,959.0,314.35,984.72,2.729437e+07,2.687719e+15,NaN,9.859619e+06,0.3612
1,2007-11-28,MUNDRAPORT,EQ,962.90,984.000000,990.000000,874.000000,885.0,314.35,941.38,4.581338e+06,4.312765e+14,NaN,1.453278e+06,0.3172
2,2007-11-29,MUNDRAPORT,EQ,893.90,909.000000,914.750000,841.000000,887.0,314.35,888.09,5.124121e+06,4.550658e+14,NaN,1.069678e+06,0.2088
3,2007-11-30,MUNDRAPORT,EQ,884.20,890.000000,958.000000,890.000000,929.0,314.35,929.17,4.609762e+06,4.283257e+14,NaN,1.260913e+06,0.2735
4,2007-12-01,MUNDRAPORT,EQ,896.65,906.583333,970.333333,900.666667,946.0,314.35,941.33,4.065665e+06,3.813904e+14,NaN,1.112650e+06,0.2737


In [3]:
from analyzer.analyzer import adf_test, kpss_test, get_volatility_stats

# 1. Run Stationarity Tests
print("--- Stationarity Analysis ---")
adf_results = adf_test(df['Close'])
kpss_results = kpss_test(df['Close'])

print(f"ADF Statistic: {adf_results['statistic']:.4f} (p-value: {adf_results['p-value']:.4f})")
print(f"KPSS Statistic: {kpss_results['statistic']:.4f} (p-value: {kpss_results['p-value']:.4f})")

# 2. Check Volatility
vol = get_volatility_stats(df)
print(f"\n--- Volatility Stats ---")
print(f"Daily Volatility: {vol['Daily Volatility']:.2%}")
print(f"Annual Volatility: {vol['Annual Volatility']:.2%}")

--- Stationarity Analysis ---
ADF Statistic: -2.4909 (p-value: 0.1177)
KPSS Statistic: 1.6090 (p-value: 0.0100)

--- Volatility Stats ---
Daily Volatility: 6.25%
Annual Volatility: 99.25%


L:\TU_Uni\ProgrammingCourse_python\TS_Analysis_Project\src\analyzer\analyzer.py:12: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  res = kpss(series.dropna(), regression='c', nlags="auto")


In [4]:
from analyzer.analyzer import decompose_signal

# Decompose the 'Close' price (period=365 for annual cycles)
decomposition = decompose_signal(df, column='Close', period=365)

# Plot the components
plt.rcParams['figure.figsize'] = (12, 10)
decomposition.plot()
plt.show()

NameError: name 'plt' is not defined

In [15]:
import sys, os
# This ensures we look in the 'src' folder
sys.path.append(os.path.abspath("../src"))

# Import the specific functions
from analyzer.visualizer import plot_seasonal_heatmap, plot_pacf_acf, plot_rolling_stats

ImportError: cannot import name 'plot_seasonal_heatmap' from 'analyzer.visualizer' (L:\TU_Uni\ProgrammingCourse_python\TS_Analysis_Project\src\analyzer\visualizer.py)

In [7]:
# 1. Load Raw Data
df_raw = pd.read_csv("../data/ADANIPORTS.csv")
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

# 2. Fill Gaps (Missing Dates) and Interpolate
# This ensures the timeline is continuous
df_filled = fill_gaps(df_raw, column='Close')

# 3. Handle Outliers
# Replaces "crazy" price spikes with the median
df_clean = handle_outliers(df_filled, column='Close')

print("Data Cleaning Complete!")
df_clean.head()

ValueError: cannot reindex on an axis with duplicate labels

## Is the Data Stationary?
Before we can predict stock prices, we must check if the data is "Stationary" 
(meaning its mean and variance don't change over time). 
We use the **Augmented Dickey-Fuller (ADF) Test**.

In [ ]:
stats = adf_test(df_clean['Close'])

print("--- ADF Test Results ---")
for key, value in stats.items():
    print(f"{key}: {value:.4f}")

if stats['p-value'] < 0.05:
    print("\nResult: Data is Stationary (Ready for forecasting)")
else:
    print("\nResult: Data is NOT Stationary (Needs differencing)")

ValueError: Could not interpret value `Close` for `y`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## Trend and Seasonality
Now we split the stock price into three parts:
1. **Trend**: The long-term direction.
2. **Seasonal**: Patterns that repeat every year.
3. **Resid (Noise)**: Random fluctuations.

In [ ]:
# Decompose the 'Close' price
# We use 365 because it's daily data
result = decompose_signal(df_clean, column='Close')

# Plot the results
plt.rcParams['figure.figsize'] = (12, 10)
result.plot()
plt.show()